# Sistem Prediksi Risiko Penyakit Ginjal Kronis

**Topik: Sistem Prediksi Risiko Penyakit Ginjal Kronis Menggunakan Perbandingan Algoritma Machine Learning Berbasis Web**

Sistem ini adalah demo skrining edukatif berbasis machine learning untuk Penyakit Ginjal Kronis (*Chronic Kidney Disease* / CKD). Proyek ini melatih beberapa model pengklasifikasi *scikit-learn* menggunakan dataset UCI CKD, membandingkan algoritma Logistic Regression, Decision Tree, Random Forest, dan SVC.

Model terbaik akan dipilih berdasarkan skor F1 yang divalidasi silang (*cross-validated*). Sistem menyimpan *pipeline* tunggal yang mencakup pra-pemrosesan dan model, serta menyajikan prediksi melalui API FastAPI dan antarmuka web statis.

*Catatan Keselamatan: Sistem ini bukan untuk diagnosis medis.*

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import json

# Menambahkan root folder ke sys.path agar bisa mengimpor module dari src dan app
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.train import load_dataset, candidate_models, compare_models, train_and_save
from src.preprocess import FEATURE_FIELDS, TARGET_COLUMN
from app.services.predictor import Predictor

In [ ]:
# Memuat dataset utama
# Catatan: program utama memakai data/raw/ckd.csv
data_path = ROOT / "data" / "raw" / "ckd.csv"

# Menggunakan fungsi bawaan dari train.py
X, y = load_dataset(data_path)
print(f"Dataset berhasil dimuat dengan ukuran X: {X.shape}, y: {y.shape}")

In [ ]:
# Menampilkan beberapa baris pertama data fitur
X.head()

In [ ]:
# Melihat jumlah missing values pada masing-masing fitur
missing_summary = X.isnull().sum()
missing_summary[missing_summary > 0]

## Penjelasan Fitur dan Target

Dataset memiliki 24 atribut CKD dan 1 kelas target (`ckd` atau `notckd`).

**Fitur Numerik:**
* `age`, `blood_pressure`, `specific_gravity`, `albumin`, `sugar`, `blood_glucose_random`, `blood_urea`, `serum_creatinine`, `sodium`, `potassium`, `hemoglobin`, `packed_cell_volume`, `white_blood_cell_count`, `red_blood_cell_count`

**Fitur Kategorikal:**
* `red_blood_cells`, `pus_cell`, `pus_cell_clumps`, `bacteria`, `hypertension`, `diabetes_mellitus`, `coronary_artery_disease`, `appetite`, `pedal_edema`, `anemia`

In [ ]:
# Training semua model dan memilih model terbaik menggunakan fungsi dari src.train
# Fungsi ini juga menyimpan artifact pipeline.joblib, metrics.json, dsb ke app/artifacts/
print("Memulai proses training dan evaluasi model...")
results = train_and_save(random_state=42, data_path=data_path, artifact_dir=ROOT / "app" / "artifacts")
print("Proses training selesai!")

In [ ]:
# Menampilkan performa (leaderboard) model-model berdasarkan cross-validated F1-score
leaderboard = results['metrics']['cv_scores']
df_leaderboard = pd.DataFrame(leaderboard).T[['mean_f1', 'std_f1']].sort_values(by='mean_f1', ascending=False)
df_leaderboard

In [ ]:
# Evaluasi model terbaik pada hold-out test set
print(f"Model terbaik yang dipilih: {results['selected_model']}")
print("\nMetrik Evaluasi Test Set:")
eval_metrics = {k: v for k, v in results['metrics'].items() if k not in ['selected_model', 'cv_scores', 'confusion_matrix', 'roc_auc']}
for k, v in eval_metrics.items():
    print(f"{k}: {v:.4f}")

print("\nConfusion Matrix:")
for row in results['metrics']['confusion_matrix']:
    print(row)

if 'roc_auc' in results['metrics']:
    print(f"\nROC-AUC Score: {results['metrics']['roc_auc']:.4f}")

In [ ]:
# Memuat feature importance dari artifact yang telah dibuat
feature_importance_path = Path(results['feature_importance_path'])
with open(feature_importance_path, 'r') as f:
    feature_importance_data = json.load(f)

df_importance = pd.DataFrame(feature_importance_data)
df_importance.head(10)

In [ ]:
# Menyimulasikan prediksi untuk contoh data
predictor = Predictor(ROOT / "app" / "artifacts" / "pipeline.joblib")

sample_input = {
  "age": 55,
  "blood_pressure": 80,
  "specific_gravity": 1.02,
  "albumin": 0,
  "sugar": 0,
  "red_blood_cells": "normal",
  "pus_cell": "normal",
  "pus_cell_clumps": "notpresent",
  "bacteria": "notpresent",
  "blood_glucose_random": 120,
  "blood_urea": 40,
  "serum_creatinine": 1.2,
  "sodium": 137,
  "potassium": 4.5,
  "hemoglobin": 14.5,
  "packed_cell_volume": 45,
  "white_blood_cell_count": 8000,
  "red_blood_cell_count": 5.2,
  "hypertension": "no",
  "diabetes_mellitus": "no",
  "coronary_artery_disease": "no",
  "appetite": "good",
  "pedal_edema": "no",
  "anemia": "no"
}

prediction = predictor.predict(sample_input)
print(f"Prediksi: {prediction['prediction']}")
print(f"Probabilitas: {prediction['probability']:.4f}")

## Cara Menjalankan Web App

Untuk menjalankan antarmuka web, buka terminal/PowerShell di direktori root project, lalu jalankan:

```powershell
uvicorn app.main:app --reload --port 8000
```

Lalu buka `http://127.0.0.1:8000` di *browser*.

## Catatan Keselamatan (Safety Note)

Sistem ini adalah **screening edukatif berbasis machine learning**, dan bukan alat diagnostik medis. Hasil prediksi dari sistem ini tidak boleh digunakan sebagai pengganti diagnosis, pemeriksaan, atau perawatan medis profesional.

In [ ]:
from pathlib import Path

artifact_dir = Path("app/artifacts")
expected_files = [
    "pipeline.joblib",
    "metrics.json",
    "model_card.json",
    "feature_importance.json",
]

print("Validasi keberadaan file artifacts:\n")
for filename in expected_files:
    path = artifact_dir / filename
    print(filename, "OK" if path.exists() else "MISSING")